# 09. Preprocessing (전처리)

## 목적
슬라이딩 윈도우 통합 데이터의 결측치 처리

## 입력
- `sliding_window_merged.csv`: 슬라이딩 윈도우 통합 데이터

## 출력
- `preprocessed.csv`: 결측치 처리된 데이터

## 전처리 전략 (임상적 관점)
| 피처 | 결측률 | 전략 | 근거 |
|------|--------|------|------|
| Vitals (hr, rr, spo2, sbp, dbp, mbp) | 1~3% | FFill → BFill → Median | 연속 모니터링, 결측=일시 중단 |
| Temp | 71% | FFill(limit=6) → 정상값(36.8) | 4-8시간 간격 측정 |
| Labs (creatinine, wbc, platelets) | 8~10% | FFill(limit=24) → Median | 일단위 검사 |
| Labs (potassium, sodium) | 9% | FFill(limit=12) → Median | 전해질, 변동 빠름 |
| Lactate | 72% | FFill(limit=12) → 정상값(1.2) + Missing Flag | 측정 자체가 중증도 지표 |
| GCS | 9~10% | FFill → BFill → Median | 의식 상태 유지 가정 |
| Urine | 24% | Median | Foley 유무 |

## 드랍 피처
- sao2 (91%): spo2와 중복
- ph (68%): ABGA에서만 측정
- bilirubin (66%): 예측 기여도 낮음

## 생성 플래그
- lactate_missing: 젖산 측정 여부
- abga_checked: ABGA 검사 시행 여부 (ph 또는 sao2 측정)

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

print("=== 09. Preprocessing 시작 ===")

## Step 1: 데이터 로드

In [ ]:
print("Step 1: 데이터 로드")

df = pd.read_csv(
    os.path.join(INPUT_DIR, 'sliding_window_merged.csv'),
    parse_dates=['observation_start', 'observation_end']
)
print(f"✓ 데이터 로드 완료: {len(df):,} rows")
print(f"\n고유 환자: {df['stay_id'].nunique():,}명")
print(f"\n고유 stay: {df['stay_id'].nunique():,}개")

# 환자별, 시간순 정렬 (FFill 전 필수)
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)
print(f"✓ 정렬 완료")

In [ ]:
# 피처 컬럼 정의
vital_cols = ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp']  # temp 별도
vital_stat_cols = ['hr_max', 'rr_max', 'spo2_min', 'sbp_min']  # 08에서 생성
lab_cols = ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium']  # lactate 별도
gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_cols = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']
drop_cols = ['sao2', 'ph', 'bilirubin']

print(f"\n피처 구성:")
print(f"  Vitals: {vital_cols}")
print(f"  Vital Stats: {vital_stat_cols}")
print(f"  Labs: {lab_cols}")
print(f"  GCS: {gcs_cols}")
print(f"  Urine: {urine_cols}")
print(f"  드랍 예정: {drop_cols}")

## Step 2: 결측치 현황 (처리 전)

In [ ]:
print("Step 2: 결측치 현황 (처리 전)")

all_cols = vital_cols + ['temp'] + vital_stat_cols + lab_cols + ['lactate'] + gcs_cols + urine_cols

print("\n=== 피처별 결측률 ===")
for col in all_cols:
    if col in df.columns:
        missing = df[col].isna().mean() * 100
        print(f"  {col}: {missing:.1f}%")
    else:
        print(f"  {col}: (컬럼 없음)")

## Step 3: ABGA 플래그 생성 & 고결측 피처 드랍

### ABGA (동맥혈 가스 분석)
- ph, sao2는 ABGA에서만 측정
- **검사 시행 여부 자체가 중증도 지표** (호흡부전/쇼크 의심 시 시행)
- 드랍 전에 플래그 먼저 생성!

### 드랍 피처
- **sao2** (91%): spo2와 중복
- **ph** (68%): 결측 너무 높음
- **bilirubin** (66%): 예측 기여도 낮음

In [ ]:
print("Step 3: ABGA 플래그 생성 & 고결측 피처 드랍")

# --- ABGA checked 플래그 먼저 생성 (드랍 전!) ---
ph_measured = df['ph'].notna() if 'ph' in df.columns else pd.Series(False, index=df.index)
sao2_measured = df['sao2'].notna() if 'sao2' in df.columns else pd.Series(False, index=df.index)
df['abga_checked'] = (ph_measured | sao2_measured).astype(int)

abga_count = df['abga_checked'].sum()
abga_rate = df['abga_checked'].mean() * 100
print(f"  ✓ abga_checked 생성: {abga_count:,}건 ({abga_rate:.1f}%)")

# --- 이제 드랍 ---
for col in drop_cols:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"  ✓ {col} 드랍")

print(f"\n현재 컬럼 수: {len(df.columns)}개")

## Step 4: Vital Signs 결측치 처리

### 임상적 근거
- ICU에서 연속 모니터링 → 결측 = 센서 일시 탈착
- **hr, rr, spo2, sbp, dbp, mbp**: FFill → BFill → Median
- **temp**: 4-8시간 간격 수동 측정 → FFill(limit=6) → 정상값(36.8)

In [ ]:
print("Step 4: Vital Signs 결측치 처리")

# --- 4-1: HR, RR, SpO2, SBP, DBP, MBP ---
print("\n[4-1] HR, RR, SpO2, SBP, DBP, MBP")

for col in vital_cols:
    if col not in df.columns:
        continue
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill().bfill()
    after_fill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before:,} → {after_fill:,} → 0")

# --- 4-2: Temp (고결측) ---
print("\n[4-2] Temp")
col = 'temp'
if col in df.columns:
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill(limit=6)
    after_ffill = df[col].isna().sum()
    
    normal_temp = 36.8
    df[col] = df[col].fillna(normal_temp)
    
    print(f"  {col}: {before:,} → FFill(6) → {after_ffill:,} → Normal({normal_temp}) → 0")

print("\n✓ Vital Signs 처리 완료")

## Step 5: Vital Stats 결측치 처리

08_sliding_window_merge에서 생성된 통계 피처:
- hr_max, rr_max, spo2_min, sbp_min

In [ ]:
print("Step 5: Vital Stats 결측치 처리")

for col in vital_stat_cols:
    if col not in df.columns:
        print(f"  ⚠️ {col} 없음 (08에서 생성 필요)")
        continue
    
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill().bfill()
    after_fill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before:,} → {after_fill:,} → 0")

print("\n✓ Vital Stats 처리 완료")

## Step 6: Lab Values 결측치 처리 (Lactate 제외)

### 임상적 근거
- Lab은 의사 오더 기반 비정기 검사
- **creatinine, wbc, platelets**: FFill(limit=24) → Median
- **potassium, sodium**: FFill(limit=12) → Median (전해질, 변동 빠름)

In [ ]:
print("Step 6: Lab Values 결측치 처리")

# --- 6-1: creatinine, wbc, platelets ---
print("\n[6-1] creatinine, wbc, platelets (FFill limit=24)")
lab_24h = ['creatinine', 'wbc', 'platelets']

for col in lab_24h:
    if col not in df.columns:
        continue
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill(limit=24)
    after_ffill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before:,} → {after_ffill:,} → 0")

# --- 6-2: potassium, sodium ---
print("\n[6-2] potassium, sodium (FFill limit=12)")
lab_12h = ['potassium', 'sodium']

for col in lab_12h:
    if col not in df.columns:
        continue
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill(limit=12)
    after_ffill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before:,} → {after_ffill:,} → 0")

print("\n✓ Lab Values 처리 완료")

## Step 7: Lactate 결측치 처리

### 임상적 근거 (핵심)
- **72% 결측 = 대부분 "측정 안 함"**
- 쇼크/패혈증 의심 시에만 측정 → **측정 여부 자체가 중증도 지표**
- 결측 = 의료진이 "안정적"이라 판단

### 처리 전략
1. `lactate_missing` 플래그 생성
2. FFill(limit=12)
3. 남은 결측 → 정상 상한값 (1.2 mmol/L)

In [ ]:
print("Step 7: Lactate 결측치 처리")

col = 'lactate'
if col in df.columns:
    before = df[col].isna().sum()
    
    # 1) Missing Flag
    df['lactate_missing'] = df[col].isna().astype(int)
    print(f"  ✓ lactate_missing 생성")
    print(f"    측정: {(df['lactate_missing']==0).sum():,}, 미측정: {(df['lactate_missing']==1).sum():,}")
    
    # 2) FFill
    df[col] = df.groupby('stay_id')[col].ffill(limit=12)
    after_ffill = df[col].isna().sum()
    
    # 3) 정상값
    normal_lactate = 1.2
    df[col] = df[col].fillna(normal_lactate)
    
    print(f"  {col}: {before:,} → FFill(12) → {after_ffill:,} → Normal({normal_lactate}) → 0")

print("\n✓ Lactate 처리 완료")

## Step 8: GCS 결측치 처리

### 임상적 근거
- 의식 명료하면 자주 측정 안 함 → 결측 = 이전 상태 유지

In [ ]:
print("Step 8: GCS 결측치 처리")

for col in gcs_cols:
    if col not in df.columns:
        continue
    before = df[col].isna().sum()
    
    df[col] = df.groupby('stay_id')[col].ffill().bfill()
    after_fill = df[col].isna().sum()
    
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    
    print(f"  {col}: {before:,} → {after_fill:,} → 0")

print("\n✓ GCS 처리 완료")

## Step 9: Urine 결측치 처리

### 임상적 근거
- Foley catheter 없으면 측정 불가

In [ ]:
print("Step 9: Urine 결측치 처리")

for col in ['urine_ml_6h', 'urine_ml_kg_hr_avg']:
    if col not in df.columns:
        continue
    before = df[col].isna().sum()
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col}: {before:,} → Median({median_val:.2f}) → 0")

col = 'oliguria_flag'
if col in df.columns:
    before = df[col].isna().sum()
    df[col] = df[col].fillna(0)
    print(f"  {col}: {before:,} → 0 (핍뇨 아님) → 0")

print("\n✓ Urine 처리 완료")

## Step 10: 결측치 처리 완료 확인

In [ ]:
print("Step 10: 결측치 처리 완료 확인")

# 처리 대상 피처
check_cols = (
    vital_cols + ['temp'] + 
    [c for c in vital_stat_cols if c in df.columns] +
    lab_cols + ['lactate'] + 
    gcs_cols + urine_cols
)

print("\n=== 결측 확인 ===")
total_missing = 0
for col in check_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        total_missing += missing
        if missing > 0:
            print(f"  ✗ {col}: {missing:,}")

if total_missing == 0:
    print(f"  ✓ 모든 피처 결측 0건!")
else:
    print(f"\n⚠️ 총 결측: {total_missing:,}건")

## Step 11: 저장

In [ ]:
print("Step 11: 저장")

output_path = os.path.join(OUTPUT_DIR, 'preprocessed.csv')
df.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: preprocessed.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df):,}개")
print(f"  - 컬럼 수: {len(df.columns)}개")

In [ ]:
print("\n=== 저장된 컬럼 목록 ===")

print("\n[ID/시간]")
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']
for col in id_cols:
    if col in df.columns:
        print(f"  - {col}")

print("\n[Vitals]")
for col in vital_cols + ['temp']:
    if col in df.columns:
        print(f"  - {col}")

print("\n[Vital Stats]")
for col in vital_stat_cols:
    if col in df.columns:
        print(f"  - {col}")
    else:
        print(f"  - {col} (없음)")

print("\n[Labs]")
for col in lab_cols + ['lactate']:
    if col in df.columns:
        print(f"  - {col}")

print("\n[GCS]")
for col in gcs_cols:
    if col in df.columns:
        print(f"  - {col}")

print("\n[Urine]")
for col in urine_cols:
    if col in df.columns:
        print(f"  - {col}")

print("\n[Flags]")
for col in ['lactate_missing', 'abga_checked', 'is_readmission']:
    if col in df.columns:
        print(f"  - {col}")

print("\n[Labels]")
label_cols = [col for col in df.columns if 'next_' in col]
for col in sorted(label_cols):
    print(f"  - {col}")

In [ ]:
print("\n" + "="*50)
print("=== 09. Preprocessing 완료 ===")
print("="*50)
print(f"\n샘플 수: {len(df):,}개")
print(f"고유 환자: {df['subject_id'].nunique():,}명")
print(f"고유 stay: {df['stay_id'].nunique():,}개")
print(f"\n다음 단계: 10_feature_engineering.ipynb")

---
## +a: 검증

In [ ]:
# 레이블 분포
print("=== 레이블 분포 ===")
label_cols = [col for col in df.columns if 'next_' in col]

for col in sorted(label_cols):
    pos_rate = df[col].mean() * 100
    pos_count = df[col].sum()
    print(f"  {col}: {pos_count:,} ({pos_rate:.2f}%)")

In [ ]:
# 피처 기술 통계
print("\n=== 피처 기술 통계 ===")
feature_cols = vital_cols + ['temp'] + lab_cols + ['lactate'] + ['gcs_total'] + ['urine_ml_6h']
existing = [c for c in feature_cols if c in df.columns]
print(df[existing].describe().round(2))

In [ ]:
# Missing Flag 분포
print("\n=== Missing Flag 분포 ===")

if 'lactate_missing' in df.columns:
    measured = (df['lactate_missing'] == 0).sum()
    not_measured = (df['lactate_missing'] == 1).sum()
    print(f"  lactate_missing:")
    print(f"    측정 (0): {measured:,} ({measured/len(df)*100:.1f}%)")
    print(f"    미측정 (1): {not_measured:,} ({not_measured/len(df)*100:.1f}%)")

if 'abga_checked' in df.columns:
    checked = (df['abga_checked'] == 1).sum()
    not_checked = (df['abga_checked'] == 0).sum()
    print(f"\n  abga_checked:")
    print(f"    검사함 (1): {checked:,} ({checked/len(df)*100:.1f}%)")
    print(f"    검사안함 (0): {not_checked:,} ({not_checked/len(df)*100:.1f}%)")

if 'is_readmission' in df.columns:
    checked = (df['is_readmission'] == 1).sum()
    not_checked = (df['is_readmission'] == 0).sum()
    print(f"\n  is_readmission:")
    print(f"    검사함 (1): {checked:,} ({checked/len(df)*100:.1f}%)")
    print(f"    검사안함 (0): {not_checked:,} ({not_checked/len(df)*100:.1f}%)")

In [ ]:
# 실제 측정된 lactate 분포
print("\n=== Lactate 실제 측정값 분포 ===")
if 'lactate_missing' in df.columns:
    lactate_measured = df[df['lactate_missing'] == 0]['lactate']
    print(f"측정된 샘플: {len(lactate_measured):,}개")
    print(lactate_measured.describe().round(2))